# Bayesian Networks using Python (pgmpy)

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand Bayesian networks structure and semantics
- Build Bayesian networks in Python (a small pure-Python implementation that always runs, plus the equivalent pgmpy code for when that library is installed)
- Implement exact inference by enumeration
- Apply Bayesian networks to real-world decision-making problems

## 🔗 Prerequisites

- ✅ Understanding of probability and conditional independence
- ✅ Python 3.8+ installed
- ✅ (Optional) pgmpy library (`pip install pgmpy`) — NOT required: every example runs on the pure-Python implementation below

---

This notebook covers practical activities from **Course 02, Unit 3**:
- Building Bayesian networks in Python (pure-Python implementation, with the pgmpy library API shown alongside)
- Implementing inference algorithms for Bayesian networks (inference by enumeration)

---

## Introduction to Bayesian Networks

**Bayesian Networks** are probabilistic graphical models that represent conditional dependencies among variables using directed acyclic graphs (DAGs).

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# pgmpy is a popular Bayesian-network library. It is OPTIONAL here:
# everything below runs on a small pure-Python implementation, and the
# pgmpy equivalents run additionally whenever the library is installed.

try:
    from pgmpy.models import BayesianNetwork
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination
    print("✅ pgmpy imported successfully — library examples will run too!")
    HAS_PGMPY = True
except ImportError:
    print("⚠️  pgmpy not installed (optional). Install with: pip install pgmpy")
    print("   No problem: every example below runs on the pure-Python")
    print("   implementation in the next cell.")
    HAS_PGMPY = False


⚠️  pgmpy not installed (optional). Install with: pip install pgmpy
   No problem: every example below runs on the pure-Python
   implementation in the next cell.


In [2]:
# A minimal discrete Bayesian network with inference by enumeration.
# ~50 lines of pure Python - enough to really SEE how BN inference works.

from itertools import product

class DiscreteBayesNet:
    """Discrete Bayesian network + exact inference by enumeration."""

    def __init__(self, name):
        self.name = name
        self.variables = {}   # var -> list of states
        self.parents = {}     # var -> list of parent variable names
        self.cpts = {}        # var -> {tuple(parent states): {state: prob}}
        print(f"✅ Created Bayesian network: {name}")

    def add_variable(self, name, states, parents=None, cpt=None):
        """Add a variable with its CPT: cpt[(parent values)][state] = prob.
        Root variables use the empty tuple () as key."""
        parents = parents or []
        self.variables[name] = list(states)
        self.parents[name] = parents
        self.cpts[name] = cpt
        suffix = f", parents={parents}" if parents else " (root)"
        print(f"  ➕ Added variable: {name} states={states}{suffix}")

    def prob(self, var, value, assignment):
        """P(var = value | its parents' values in this assignment)"""
        key = tuple(assignment[p] for p in self.parents[var])
        return self.cpts[var][key][value]

    def joint(self, assignment):
        """Chain rule: P(full assignment) = product of P(var | parents)"""
        p = 1.0
        for var in self.variables:
            p *= self.prob(var, assignment[var], assignment)
        return p

    def query(self, var, evidence=None):
        """P(var | evidence), by summing the joint over all hidden variables.
        This is 'inference by enumeration' - simple and exact (but exponential;
        libraries like pgmpy use faster algorithms such as Variable Elimination)."""
        evidence = evidence or {}
        hidden = [v for v in self.variables if v != var and v not in evidence]
        totals = {}
        for target_state in self.variables[var]:
            total = 0.0
            for combo in product(*(self.variables[h] for h in hidden)):
                assignment = dict(evidence)
                assignment[var] = target_state
                assignment.update(zip(hidden, combo))
                total += self.joint(assignment)
            totals[target_state] = total
        norm = sum(totals.values())
        return {s: t / norm for s, t in totals.items()}

def print_dist(title, dist):
    print(f"   {title}")
    for state, p in dist.items():
        print(f"      {state:<5} {p:.4f}")

print("✅ DiscreteBayesNet ready (pure Python, no libraries needed)")

✅ DiscreteBayesNet ready (pure Python, no libraries needed)


## Part 1: Building a Simple Bayesian Network

Let's create a simple medical diagnosis Bayesian network.


In [3]:
# Example: Medical Diagnosis Network
# Structure: Disease -> Symptom
# P(Disease=Yes) = 0.1 (10% of patients have the disease)
# P(Symptom=Yes | Disease=Yes) = 0.9, P(Symptom=Yes | Disease=No) = 0.2

print("=" * 60)
print("Bayesian Network: Medical Diagnosis")
print("=" * 60)

medical = DiscreteBayesNet("Medical Diagnosis")

# Prior for Disease (root variable -> CPT key is the empty tuple)
medical.add_variable('Disease', ['No', 'Yes'],
                     cpt={(): {'No': 0.9, 'Yes': 0.1}})

# Symptom depends on Disease
medical.add_variable('Symptom', ['No', 'Yes'], parents=['Disease'],
                     cpt={('No',):  {'No': 0.8, 'Yes': 0.2},   # healthy: 20% show symptom
                          ('Yes',): {'No': 0.1, 'Yes': 0.9}})  # diseased: 90% show symptom

print("\nStructure: Disease -> Symptom")
print("Chain rule check: P(Disease=Yes, Symptom=Yes) =", end=" ")
print(f"{medical.joint({'Disease': 'Yes', 'Symptom': 'Yes'}):.4f}  (= 0.1 x 0.9)")

# The same model in pgmpy (runs only if the library is installed)
if HAS_PGMPY:
    model = BayesianNetwork([('Disease', 'Symptom')])
    cpd_disease = TabularCPD(variable='Disease', variable_card=2,
                             values=[[0.9],   # P(Disease = No)  = 0.9
                                     [0.1]],  # P(Disease = Yes) = 0.1
                             state_names={'Disease': ['No', 'Yes']})
    cpd_symptom = TabularCPD(variable='Symptom', variable_card=2,
                             evidence=['Disease'], evidence_card=[2],
                             values=[[0.8, 0.1],   # P(Symptom=No | Disease=No/Yes)
                                     [0.2, 0.9]],  # P(Symptom=Yes | Disease=No/Yes)
                             state_names={'Symptom': ['No', 'Yes'],
                                          'Disease': ['No', 'Yes']})
    model.add_cpds(cpd_disease, cpd_symptom)
    print(f"\npgmpy version of the same model — valid: {model.check_model()}")
else:
    print("\n(pgmpy not installed — the pure-Python model above is the one we use.)")

Bayesian Network: Medical Diagnosis
✅ Created Bayesian network: Medical Diagnosis
  ➕ Added variable: Disease states=['No', 'Yes'] (root)
  ➕ Added variable: Symptom states=['No', 'Yes'], parents=['Disease']

Structure: Disease -> Symptom
Chain rule check: P(Disease=Yes, Symptom=Yes) = 0.0900  (= 0.1 x 0.9)

(pgmpy not installed — the pure-Python model above is the one we use.)


## Part 2: Bayesian Inference

Now let's perform inference: given evidence (symptom), what's the probability of disease?


In [4]:
# Inference: given evidence (symptom), what is the probability of disease?

print("=" * 60)
print("Bayesian Inference Examples:")
print("=" * 60)

# Query 1: Prior probability of disease (no evidence)
print("\n1. Prior Probability of Disease:")
prior = medical.query('Disease')
print_dist("P(Disease):", prior)

# Query 2: Posterior probability given the symptom is observed
print("\n2. Posterior Probability: P(Disease | Symptom = Yes)")
posterior = medical.query('Disease', evidence={'Symptom': 'Yes'})
print_dist("P(Disease | Symptom=Yes):", posterior)

# Query 3: Marginal probability of the symptom
print("\n3. Marginal Probability: P(Symptom)")
marginal = medical.query('Symptom')
print_dist("P(Symptom):", marginal)

# Hand-check with Bayes' theorem (all computed, nothing hardcoded):
p_s_yes = 0.9 * 0.2 + 0.1 * 0.9                # P(S=Yes) = sum over Disease
p_d_given_s = (0.9 * 0.1) / p_s_yes            # P(D=Yes|S=Yes) = P(S|D)P(D)/P(S)
print("\n✔ Bayes'-theorem hand check:")
print(f"   P(Symptom=Yes) = 0.9x0.2 + 0.1x0.9 = {p_s_yes:.4f}")
print(f"   P(Disease=Yes | Symptom=Yes) = 0.1x0.9 / {p_s_yes:.4f} = {p_d_given_s:.4f}")
print(f"   Matches the enumeration result: {abs(p_d_given_s - posterior['Yes']) < 1e-12}")

# pgmpy Variable Elimination gives the same answers (if installed)
if HAS_PGMPY:
    inference = VariableElimination(model)
    print("\npgmpy (Variable Elimination) cross-check:")
    print(inference.query(variables=['Disease'], evidence={'Symptom': 'Yes'}))
else:
    print("\n(pgmpy not installed — enumeration results above are exact and complete.)")

Bayesian Inference Examples:

1. Prior Probability of Disease:
   P(Disease):
      No    0.9000
      Yes   0.1000

2. Posterior Probability: P(Disease | Symptom = Yes)
   P(Disease | Symptom=Yes):
      No    0.6667
      Yes   0.3333

3. Marginal Probability: P(Symptom)
   P(Symptom):
      No    0.7300
      Yes   0.2700

✔ Bayes'-theorem hand check:
   P(Symptom=Yes) = 0.9x0.2 + 0.1x0.9 = 0.2700
   P(Disease=Yes | Symptom=Yes) = 0.1x0.9 / 0.2700 = 0.3333
   Matches the enumeration result: True

(pgmpy not installed — enumeration results above are exact and complete.)


## Part 3: More Complex Bayesian Network

Let's build a more complex network with multiple variables.


In [5]:
# Example: the classic Alarm network (Russell & Norvig)
# Structure: Burglary -> Alarm <- Earthquake ; Alarm -> JohnCalls, MaryCalls

print("=" * 60)
print("Complex Bayesian Network: Alarm System")
print("=" * 60)

alarm_net = DiscreteBayesNet("Alarm System")

alarm_net.add_variable('Burglary',   ['No', 'Yes'], cpt={(): {'No': 0.999, 'Yes': 0.001}})
alarm_net.add_variable('Earthquake', ['No', 'Yes'], cpt={(): {'No': 0.998, 'Yes': 0.002}})

# Alarm depends on both Burglary and Earthquake: cpt[(burglary, earthquake)]
alarm_net.add_variable('Alarm', ['No', 'Yes'], parents=['Burglary', 'Earthquake'],
    cpt={('No',  'No'):  {'No': 0.999, 'Yes': 0.001},
         ('No',  'Yes'): {'No': 0.71,  'Yes': 0.29},
         ('Yes', 'No'):  {'No': 0.06,  'Yes': 0.94},
         ('Yes', 'Yes'): {'No': 0.05,  'Yes': 0.95}})

alarm_net.add_variable('JohnCalls', ['No', 'Yes'], parents=['Alarm'],
    cpt={('No',):  {'No': 0.95, 'Yes': 0.05},
         ('Yes',): {'No': 0.10, 'Yes': 0.90}})

alarm_net.add_variable('MaryCalls', ['No', 'Yes'], parents=['Alarm'],
    cpt={('No',):  {'No': 0.99, 'Yes': 0.01},
         ('Yes',): {'No': 0.30, 'Yes': 0.70}})

# Inference: both neighbors call - was it a burglary?
print("\nInference: P(Burglary | JohnCalls=Yes, MaryCalls=Yes)")
result = alarm_net.query('Burglary', evidence={'JohnCalls': 'Yes', 'MaryCalls': 'Yes'})
print_dist("P(Burglary | both call):", result)

print("\nInference: P(Alarm | JohnCalls=Yes, MaryCalls=Yes)")
result_alarm = alarm_net.query('Alarm', evidence={'JohnCalls': 'Yes', 'MaryCalls': 'Yes'})
print_dist("P(Alarm | both call):", result_alarm)

print(f"\n💡 Interpretation (computed above): even with BOTH neighbors calling,")
print(f"   P(Burglary=Yes) is only {result['Yes']:.3f} — because burglaries are rare")
print(f"   (prior 0.001) and the alarm can also be triggered by earthquakes.")

Complex Bayesian Network: Alarm System
✅ Created Bayesian network: Alarm System
  ➕ Added variable: Burglary states=['No', 'Yes'] (root)
  ➕ Added variable: Earthquake states=['No', 'Yes'] (root)
  ➕ Added variable: Alarm states=['No', 'Yes'], parents=['Burglary', 'Earthquake']
  ➕ Added variable: JohnCalls states=['No', 'Yes'], parents=['Alarm']
  ➕ Added variable: MaryCalls states=['No', 'Yes'], parents=['Alarm']

Inference: P(Burglary | JohnCalls=Yes, MaryCalls=Yes)
   P(Burglary | both call):
      No    0.7158
      Yes   0.2842

Inference: P(Alarm | JohnCalls=Yes, MaryCalls=Yes)
   P(Alarm | both call):
      No    0.2393
      Yes   0.7607

💡 Interpretation (computed above): even with BOTH neighbors calling,
   P(Burglary=Yes) is only 0.284 — because burglaries are rare
   (prior 0.001) and the alarm can also be triggered by earthquakes.


## Summary

### Key Concepts:
1. **Bayesian Networks**: Graphical models representing conditional dependencies
2. **CPDs / CPTs (Conditional Probability Distributions/Tables)**: Define relationships between variables
3. **Inference**: Compute posterior probabilities given evidence
4. **Inference by Enumeration**: The simple exact algorithm we implemented here — sum the joint distribution over hidden variables (libraries like pgmpy speed this up with Variable Elimination)

### Applications:
- Medical diagnosis
- Risk assessment
- Decision support systems
- Natural language processing

**Reference:** Course 02, Unit 3: "Building Bayesian networks using Python libraries (pgmpy)" and "Implementing inference algorithms for Bayesian networks" — implemented here in pure Python so it runs everywhere, with the equivalent pgmpy code included for environments that have the library installed.